# **K-Means Clustering — Seller Segmentation**

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.preprocessing import PowerTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

import plotly.express as px
import plotly.io as pio

from scipy.stats import kruskal


# =========================
# Notebook display setting
# =========================
pio.renderers.default = "notebook_connected"


# =========================
# Directory configuration
# =========================
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

NOTEBOOK_OUTPUT_DIR = PROJECT_DIR /  "outputs"

NOTEBOOK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# =========================
# Input file
# =========================
INPUT_FILE = NOTEBOOK_OUTPUT_DIR / "seller_scoring_bgnbd_gammagamma_production.csv"

# =========================
# Load data
# =========================
df = pd.read_csv(INPUT_FILE)

print(f"Loaded data from: {INPUT_FILE}")
print(f"Shape: {df.shape}")
display(df.head())

Loaded data from: e:\marketing for DS\marketing-driven-data-unlock\outputs\seller_scoring_bgnbd_gammagamma_production.csv
Shape: (2248, 25)


,seller_id,frequency,recency,T,monetary_value,total_orders_cal,total_gmv_cal,historical_aov_cal,prob_alive,expected_avg_gmv,...,expected_orders_60d,activity_score_60d,p_purchase_approx_60d,predicted_gmv_60d,expected_commission_60d,expected_orders_90d,activity_score_90d,p_purchase_approx_90d,predicted_gmv_90d,expected_commission_90d
0,0015a82c2db000af6aaaf3ae2ecb0532,2,21.416308,216.071458,895.000000,3,2685.00,895.000000,0.099162,894.276630,...,0.065367,0.065367,0.063276,58.455924,8.768389,0.097420,0.097420,0.092825,87.120216,13.068032
1,001cca7ae9ae17fb1caed9dfb1094831,191,447.554861,450.204109,125.739424,192,24116.13,125.604844,0.997701,125.777572,...,25.038213,25.038213,1.000000,3149.245635,472.386845,37.397244,37.397244,1.000000,4703.734589,705.560188
2,002100f778ceb8431b7a1020ff7ab48f,49,210.498519,228.957963,24.626531,50,1216.60,24.332000,0.853171,24.798646,...,10.712664,10.712664,0.999978,265.659572,39.848936,15.948371,15.948371,1.000000,395.498008,59.324701
3,003554e2dce176b5555353e4f3555ac8,0,0.000000,136.713588,0.000000,1,120.00,120.000000,1.000000,120.000000,...,0.187662,0.187662,0.171106,22.519494,3.377924,0.279443,0.279443,0.243795,33.533129,5.029969
4,004c9cd9d87a3c30c522c48c4fc07416,153,444.125498,458.559317,124.838170,154,19220.23,124.806688,0.891200,124.885858,...,17.604879,17.604879,1.000000,2198.600440,329.790066,26.296725,26.296725,1.000000,3284.089090,492.613363


In [2]:
# =========================
# K-Means configuration
# =========================
HORIZON = 60
K = None

FEATURE_GROUP_NAME = f"orders_purchase_commission_{HORIZON}d"

FEATURES = [
    f"expected_orders_{HORIZON}d",
    f"p_purchase_approx_{HORIZON}d",
    f"expected_commission_{HORIZON}d",
]

print("Feature group:", FEATURE_GROUP_NAME)
print("Features:", FEATURES)


# =========================
# Validate selected features
# =========================
missing_cols = [col for col in FEATURES if col not in df.columns]

if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")


# =========================
# Prepare feature matrix
# =========================
X_raw = df[FEATURES].copy()

X_raw = X_raw.replace([np.inf, -np.inf], np.nan)
X_raw = X_raw.fillna(0)

# =========================
# Transform skewness + standardize
# =========================
transformer = PowerTransformer(
    method="yeo-johnson",
    standardize=True
)

X_scaled = transformer.fit_transform(X_raw)

# ============================================================
# Choose K using Elbow Method + Silhouette Score
# ============================================================

from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import plotly.express as px
import pandas as pd

# -------------------------
# 1. Configuration
# -------------------------
K_RANGE_START = 2
K_RANGE_END = 10          # sẽ test từ k=2 đến k=10
ELBOW_CANDIDATES = [3, 4] # 2 điểm elbow bạn muốn so sánh

n_samples = X_scaled.shape[0]

valid_k_values = list(
    range(
        K_RANGE_START,
        min(K_RANGE_END, n_samples - 1) + 1
    )
)

if len(valid_k_values) == 0:
    raise ValueError("Not enough samples to evaluate K-Means clustering.")

# -------------------------
# 2. Fit K-Means for each K
# -------------------------
k_eval_rows = []

for k in valid_k_values:
    temp_kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20,
        max_iter=300
    )
    
    temp_labels = temp_kmeans.fit_predict(X_scaled)
    
    k_eval_rows.append({
        "k": k,
        "wcss": temp_kmeans.inertia_,
        "silhouette_score": silhouette_score(X_scaled, temp_labels)
    })

k_eval = pd.DataFrame(k_eval_rows)

display(
    k_eval.style.format({
        "wcss": "{:,.2f}",
        "silhouette_score": "{:.4f}"
    })
)

# -------------------------
# 3. Plot Elbow Method
# -------------------------
fig_elbow = px.line(
    k_eval,
    x="k",
    y="wcss",
    markers=True,
    title="Elbow Method - WCSS by Number of Clusters",
    labels={
        "k": "Number of clusters (K)",
        "wcss": "WCSS / Inertia"
    }
)

candidate_plot_df = k_eval[k_eval["k"].isin(ELBOW_CANDIDATES)].copy()

fig_elbow.add_scatter(
    x=candidate_plot_df["k"],
    y=candidate_plot_df["wcss"],
    mode="markers+text",
    text=[f"k={k}" for k in candidate_plot_df["k"]],
    textposition="top center",
    name="Elbow candidates"
)

fig_elbow.update_layout(
    xaxis=dict(dtick=1),
    width=900,
    height=500
)

fig_elbow.show()

# -------------------------
# 4. Compare Silhouette only for elbow candidates
# -------------------------
candidate_scores = (
    k_eval[k_eval["k"].isin(ELBOW_CANDIDATES)]
    .copy()
    .sort_values("silhouette_score", ascending=False)
)

if candidate_scores.empty:
    raise ValueError(
        f"None of ELBOW_CANDIDATES={ELBOW_CANDIDATES} are inside valid K range={valid_k_values}."
    )

display(
    candidate_scores.style.format({
        "wcss": "{:,.2f}",
        "silhouette_score": "{:.4f}"
    })
)

fig_silhouette_candidates = px.bar(
    candidate_scores.sort_values("k"),
    x="k",
    y="silhouette_score",
    text="silhouette_score",
    title="Silhouette Score Comparison for Elbow Candidates",
    labels={
        "k": "Number of clusters (K)",
        "silhouette_score": "Silhouette Score"
    }
)

fig_silhouette_candidates.update_traces(
    texttemplate="%{text:.4f}",
    textposition="outside"
)

fig_silhouette_candidates.update_layout(
    xaxis=dict(dtick=1),
    width=700,
    height=450
)

fig_silhouette_candidates.show()

# -------------------------
# 5. Select final K
# -------------------------
best_row = candidate_scores.iloc[0]

K = int(best_row["k"])

print("=" * 60)
print("K-SELECTION RESULT")
print("=" * 60)
print(f"Elbow candidates: {ELBOW_CANDIDATES}")
print("Silhouette scores among elbow candidates:")

for _, row in candidate_scores.sort_values("k").iterrows():
    print(f"  K={int(row['k'])}: Silhouette Score = {row['silhouette_score']:.4f}")

print("-" * 60)
print(f"Selected K = {K}")
print(
    f"Reason: among the elbow candidates {ELBOW_CANDIDATES}, "
    f"K={K} has the highest Silhouette Score."
)
print("=" * 60)

# =========================
# Fit K-Means
# =========================
kmeans = KMeans(
    n_clusters=K,
    random_state=42,
    n_init=20,
    max_iter=300
)

cluster_labels = kmeans.fit_predict(X_scaled)


# =========================
# Attach cluster labels
# =========================
df_kmeans = df.copy()

df_kmeans["kmeans_feature_group"] = FEATURE_GROUP_NAME
df_kmeans["kmeans_horizon"] = HORIZON
df_kmeans["kmeans_k"] = K
df_kmeans["seller_cluster"] = cluster_labels


# =========================
# Evaluation metrics
# =========================
silhouette = silhouette_score(X_scaled, cluster_labels)
davies_bouldin = davies_bouldin_score(X_scaled, cluster_labels)
calinski_harabasz = calinski_harabasz_score(X_scaled, cluster_labels)

print(f"K-Means Feature Group: {FEATURE_GROUP_NAME}")
print(f"Horizon: {HORIZON} days")
print(f"K: {K}")
print(f"Silhouette Score: {silhouette:.4f}")
print(f"Davies-Bouldin Score: {davies_bouldin:.4f}")
print(f"Calinski-Harabasz Score: {calinski_harabasz:.4f}")

Feature group: orders_purchase_commission_60d
Features: ['expected_orders_60d', 'p_purchase_approx_60d', 'expected_commission_60d']


,k,wcss,silhouette_score
0,2,"1,582.45",0.6426
1,3,945.80,0.5295
2,4,643.30,0.4789
3,5,501.50,0.4443
4,6,429.38,0.4171
5,7,371.28,0.4011
6,8,321.33,0.4072
7,9,282.75,0.4123
8,10,254.69,0.4100


,k,wcss,silhouette_score
1,3,945.80,0.5295
2,4,643.30,0.4789


K-SELECTION RESULT
Elbow candidates: [3, 4]
Silhouette scores among elbow candidates:
  K=3: Silhouette Score = 0.5295
  K=4: Silhouette Score = 0.4789
------------------------------------------------------------
Selected K = 3
Reason: among the elbow candidates [3, 4], K=3 has the highest Silhouette Score.
K-Means Feature Group: orders_purchase_commission_60d
Horizon: 60 days
K: 3
Silhouette Score: 0.5295
Davies-Bouldin Score: 0.6834
Calinski-Harabasz Score: 6881.4411


### K Selection using Elbow Method and Silhouette Score

To determine the appropriate number of clusters for K-Means, we evaluated K values from 2 to 10 using two criteria: WCSS/Inertia and Silhouette Score.

The Elbow Method shows that WCSS decreases sharply when K increases from 2 to 3, and continues to decrease from 3 to 4. However, after K=4, the reduction in WCSS becomes more gradual, indicating that the marginal improvement from adding more clusters becomes smaller. Therefore, K=3 and K=4 were selected as the main elbow candidates.

Although K=4 produces a lower WCSS than K=3, this is expected because WCSS naturally decreases as the number of clusters increases. To choose between the two elbow candidates, we compared their Silhouette Scores. The result shows that K=3 has a higher Silhouette Score (0.5295) than K=4 (0.4789). This indicates that the cluster structure at K=3 is more compact and better separated.

Therefore, K=3 was selected as the final number of clusters. This choice balances model quality and business interpretability, allowing sellers to be segmented into three meaningful groups: dormant sellers, mid-tier engaged sellers, and high-value active sellers.

In [3]:
# =========================
# Cluster profile
# =========================
cluster_profile = (
    df_kmeans
    .groupby("seller_cluster")
    .agg(
        n_sellers=("seller_cluster", "count"),

        expected_orders_60d_mean=(f"expected_orders_{HORIZON}d", "mean"),
        expected_orders_60d_median=(f"expected_orders_{HORIZON}d", "median"),

        p_purchase_60d_mean=(f"p_purchase_approx_{HORIZON}d", "mean"),
        p_purchase_60d_median=(f"p_purchase_approx_{HORIZON}d", "median"),

        expected_commission_60d_mean=(f"expected_commission_{HORIZON}d", "mean"),
        expected_commission_60d_median=(f"expected_commission_{HORIZON}d", "median"),
        expected_commission_60d_sum=(f"expected_commission_{HORIZON}d", "sum"),
    )
    .reset_index()
)

cluster_profile["seller_share"] = (
    cluster_profile["n_sellers"] / cluster_profile["n_sellers"].sum()
)

cluster_profile["expected_commission_60d_share"] = (
    cluster_profile["expected_commission_60d_sum"]
    / cluster_profile["expected_commission_60d_sum"].sum()
)

cluster_profile = cluster_profile.sort_values(
    "expected_commission_60d_median",
    ascending=False
)

display(cluster_profile)

,seller_cluster,n_sellers,expected_orders_60d_mean,expected_orders_60d_median,p_purchase_60d_mean,p_purchase_60d_median,expected_commission_60d_mean,expected_commission_60d_median,expected_commission_60d_sum,seller_share,expected_commission_60d_share
2,2,857,13.877609,6.814062,0.985546,0.998902,283.500425,135.969629,242959.863867,0.381228,0.911484
1,1,552,1.414048,1.265127,0.700281,0.717793,37.214350,23.451205,20542.321191,0.245552,0.077066
0,0,839,0.158710,0.101584,0.136266,0.096595,3.637593,1.536463,3051.940941,0.373221,0.011450


## **Clustering Inference Insights**

**Overall Distribution Summary**

> The K-Means clustering model successfully identified three distinct seller segments with significantly different behavioral and commercial characteristics. By combining: Expected future orders, Estimated purchase probability, Projected commission value within the next 60 days, the segmentation reveals strong concentration patterns in seller value contribution across the platform.

| Cluster | Segment Label | Number of Sellers | Seller Share | Total Commission (60d) | Commission Share |
|---|---|---|---|---|---|
| 2 | High-Value Active Sellers | 857 | 38.1% | ~243,000 | 91.1% |
| 1 | Mid-Tier Engaged Sellers | 552 | 24.6% | ~20,500 | 7.7% |
| 0 | Dormant Sellers | 839 | 37.3% | ~3,050 | 1.1% |

The commission distribution is extremely concentrated: only 38% of sellers (Cluster 2) generate 91% of the projected commission. This is a classic manifestation of the Pareto Principle within a marketplace ecosystem, although at a much more extreme level than the conventional 80/20 pattern — here, the ratio is approximately 91/38.

Such a high concentration level creates significant systemic business risk. If churn rates increase within Cluster 2, the platform’s future revenue performance could be heavily impacted.

---

### **Cluster 2 — High-Value Active Sellers**

#### **_Quantitative Definition_**

| Metric | Mean | Median |
|---|---|---|
| Expected orders (60d) | 13.88 orders | 6.81 orders |
| P(purchases) | 98.6% | 99.9% |
| Expected commission (60d) | 283.5 | 135.97 |

Cluster 2 represents the platform’s most commercially valuable seller segment. Sellers in this group demonstrate:
- extremely high purchase probability,
- strong expected transaction volume,
- and dominant future commission contribution.

Despite representing only a minority of the seller base, this cluster generates almost the entire projected commission of the platform. This indicates a highly concentrated revenue structure where platform performance is heavily dependent on retaining and supporting these sellers.

#### **_Behavioral Analysis_**

Cluster 2 consists of sellers with near-perfect active probability while maintaining the highest transaction frequency across the entire seller population. According to the BG/NBD model, these sellers are considered as "fully" active, meaning their transaction history shows virtually no indication of churn behavior.

The substantial gap between the mean (13.88) and median (6.81) of expected orders suggests the presence of long-tail outliers within the cluster. A small subset of top-tier sellers likely generates extremely high projected order volumes (potentially 50–200 orders), significantly inflating the mean. A similar pattern appears in expected commission confirms a strongly right-skewed distribution even within the cluster itself.

#### **_Business Implications_**

- Cluster 2 represents the platform’s core revenue engine. Any deterioration within this group — even affecting only 5–10% of sellers — could create substantial revenue impact.
- However, not all sellers inside Cluster 2 are homogeneous. Additional internal sub-segmentation should be considered to separate top outlier sellers from the remaining population, enabling more personalized account management strategies.

---

### **Cluster 1 — Mid-Tier Engaged Sellers**

#### **_Quantitative Definition_**

| Metric | Mean | Median |
|---|---|---|
| Expected orders (60d) | 1.41 orders | 1.27 orders |
| P(purchases) | 70.0% | 71.8% |
| Expected commission (60d) | 37.2 | 23.5 |

#### **_Behavioral Analysis_**

Cluster 1 is arguably the most analytically interesting segment due to its transitional characteristics.

_Observation 1 — Moderate P(purchases) (~70%)_

The BG/NBD model estimates that approximately 70% of sellers in this group are still active, implying a churn probability of roughly 30%. This segment is neither fully active nor fully churned. Instead, it occupies a transition zone:
- sellers may recover and move upward into Cluster 2 with appropriate intervention,
- or gradually decline into Cluster 0 if neglected.

_Observation 2 — Small Mean/Median Gap_

The mean-to-median ratio is relatively small. This is significantly lower than Cluster 2, indicating that seller behavior within Cluster 1 is relatively homogeneous, without extreme outliers. The average transaction frequency (~1.3 orders per 60 days) remains low. However, since P(purchases) is still around 72%, these sellers are clearly still interacting with the platform.

The issue may therefore lie in conversion efficiency rather than seller inactivity itself.

#### **_Business Implications_**

Cluster 1 represents the platform’s most promising growth opportunity with potentially strong ROI on intervention efforts.

The cost of activation is likely much lower than acquiring entirely new sellers, while the probability of upgrading these sellers into Cluster 2 remains realistic if operational friction points are addressed.

A key business question emerges: Why is P(purchases) relatively high while order volume remains low?

Several hypotheses should be investigated:

- Sellers experience friction during listing or onboarding.
- Sellers operate seasonally and current data reflects off-season behavior.
- Sellers receive insufficient traffic or marketplace visibility.
- Sellers prioritize competing platforms over the current marketplace.

---

### **Cluster 0 — Dormant Sellers**

#### **_Quantitative Definition_**

| Metric | Mean | Median |
|---|---|---|
| Expected orders (60d) | 0.16 orders | 0.10 orders |
| P(purchases) | 13.6% | 9.7% |
| Expected commission (60d) | 3.64 | 1.54 |

#### **_Behavioral Analysis_**

Cluster 0 contains sellers that the BG/NBD model classifies as having extremely high churn probability.

A median P(purchases) of only 9.7% suggests that, probabilistically, over 90% of sellers in this cluster are effectively inactive. This means that the elapsed time since their last transaction is disproportionately long relative to their historical purchasing behavior.

An expected order value of only 0.10 orders over 60 days effectively indicates that the model predicts almost no future transactions from these sellers. Even the mean value (0.16) remains extremely low, confirming the absence of meaningful outliers within the cluster. Importantly, 37.3% of all sellers belong to this cluster, yet they contribute only 1.1% of projected commission.

In practical terms, if the entire Cluster 0 population churned immediately, the revenue impact (~3,050 commission) would roughly equal the contribution of a single average seller from Cluster 2.

#### **_Business Implications_**

The primary risk associated with Cluster 0 is not revenue loss, but operational inefficiency and opportunity cost. Resources allocated toward these sellers — customer support, marketing exposure, infrastructure, and operational maintenance — may produce substantially higher returns if redirected toward Clusters 1 or 2.

However, immediate offboarding should be avoided until additional analysis distinguishes between two important subgroups:

- Seasonal Sellers: Some sellers may only operate during seasonal periods (e.g., holidays, back-to-school periods). In such cases, low P(purchase) may simply reflect temporary inactivity during off-season periods.
- Truly Churned Sellers: Other sellers demonstrate no seasonal patterns and exhibit prolonged inactivity without explanation. These sellers are more appropriate candidates for offboarding or minimal-maintenance win-back strategies.

## **Final Conclusion**

The K-Means segmentation framework successfully identified meaningful seller groups with substantial differences in:

- future transaction behavior,
- purchase probability,
- and expected commission contribution.

This segmentation enables the platform to implement more efficient and data-driven seller management strategies, improve retention effectiveness, and optimize long-term commission growth.

In [4]:
import pandas as pd
import plotly.express as px
from IPython.display import HTML, display


# =========================
# Configuration
# =========================
HORIZON = 60

FEATURE_GROUP_NAME = f"orders_purchase_commission_{HORIZON}d"

FEATURES = [
    f"expected_orders_{HORIZON}d",
    f"p_purchase_approx_{HORIZON}d",
    f"expected_commission_{HORIZON}d",
]


# =========================
# Validate required columns
# =========================
required_cols = FEATURES + ["seller_cluster"]

missing_cols = [col for col in required_cols if col not in df_kmeans.columns]

if missing_cols:
    raise ValueError(f"Missing columns in df_kmeans: {missing_cols}")


# =========================
# Infer K dynamically
# =========================
K = df_kmeans["seller_cluster"].nunique()

# =========================
# Prepare RAW plot dataframe
# =========================
plot_raw_df = df_kmeans.copy()
plot_raw_df["seller_cluster"] = plot_raw_df["seller_cluster"].astype(str)


# =========================
# Hover columns
# =========================
hover_cols = FEATURES.copy()

if "seller_id" in plot_raw_df.columns:
    hover_cols = ["seller_id"] + hover_cols


# =========================
# Plot 1: RAW business space
# Better axis arrangement
# =========================
fig_raw = px.scatter_3d(
    plot_raw_df,
    x="expected_commission_60d",
    y="expected_orders_60d",
    z="p_purchase_approx_60d",
    color="seller_cluster",
    opacity=0.65,
    title=f"3D K-Means Clusters - Raw Business Space | {FEATURE_GROUP_NAME}, K={K}",
    hover_data=hover_cols,
)

fig_raw.update_traces(
    marker=dict(size=3)
)

fig_raw.update_layout(
    width=1100,
    height=800,
    legend_title="Seller Cluster",
    scene=dict(
        xaxis_title="Expected Commission 60d",
        yaxis_title="Expected Orders 60d",
        zaxis_title="P(Purchase) Approx 60d",
        aspectmode="manual",
        aspectratio=dict(x=1.4, y=1.2, z=0.9),
        camera=dict(
            eye=dict(x=1.8, y=1.6, z=1.1)
        )
    ),
    margin=dict(l=0, r=0, b=0, t=60)
)


# =========================
# Validate X_scaled exists
# =========================
if "X_scaled" not in globals():
    raise ValueError(
        "X_scaled not found. Please run the K-Means training cell first."
    )


# =========================
# Prepare SCALED plot dataframe
# Reuse exact X_scaled from training
# =========================
scaled_feature_names = [
    "scaled_expected_orders_60d",
    "scaled_p_purchase_approx_60d",
    "scaled_expected_commission_60d",
]

plot_scaled_df = pd.DataFrame(
    X_scaled,
    columns=scaled_feature_names,
    index=df_kmeans.index
)

plot_scaled_df["seller_cluster"] = (
    df_kmeans["seller_cluster"].astype(str)
)


# =========================
# Add raw values for hover
# =========================
for col in FEATURES:
    plot_scaled_df[col] = df_kmeans[col].values

if "seller_id" in df_kmeans.columns:
    plot_scaled_df["seller_id"] = df_kmeans["seller_id"].values


# =========================
# Hover columns for scaled plot
# =========================
hover_cols_scaled = FEATURES.copy()

if "seller_id" in plot_scaled_df.columns:
    hover_cols_scaled = ["seller_id"] + hover_cols_scaled


# =========================
# Plot 2: Scaled K-Means model space
# =========================
fig_scaled = px.scatter_3d(
    plot_scaled_df,
    x="scaled_expected_commission_60d",
    y="scaled_expected_orders_60d",
    z="scaled_p_purchase_approx_60d",
    color="seller_cluster",
    opacity=0.65,
    title=f"3D K-Means Clusters - Scaled Model Space | {FEATURE_GROUP_NAME}, K={K}",
    hover_data=hover_cols_scaled,
)

fig_scaled.update_traces(
    marker=dict(size=3)
)

fig_scaled.update_layout(
    width=1100,
    height=800,
    legend_title="Seller Cluster",
    scene=dict(
        xaxis_title="Scaled Expected Commission 60d",
        yaxis_title="Scaled Expected Orders 60d",
        zaxis_title="Scaled P(Purchase) Approx 60d",
        aspectmode="manual",
        aspectratio=dict(x=1.4, y=1.2, z=0.9),
        camera=dict(
            eye=dict(x=1.8, y=1.6, z=1.1)
        )
    ),
    margin=dict(l=0, r=0, b=0, t=60)
)


# =========================
# Display inline in notebook
# =========================
display(HTML("<h3>Raw Business Feature Space</h3>"))
display(HTML(fig_raw.to_html(
    full_html=False,
    include_plotlyjs="cdn"
)))

## **Analysis of the 3D Clustering Visualization**

### **Raw Business Feature Space**

The 3D visualization in the raw business feature space reveals a highly imbalanced data structure. Most observations are vertically compressed along the purchase probability axis, while expected orders and expected commission values exhibit extremely wide horizontal dispersion.

This pattern indicates severe right-skewness in the original feature distributions. A very small number of sellers generate exceptionally large commission values — in some cases reaching several thousand units — while the majority of sellers remain concentrated near zero.

As a consequence, the Euclidean distance calculation used by the K-Means algorithm becomes heavily dominated by the commission dimension. In practical terms, the clustering process in the raw feature space is disproportionately influenced by seller commission magnitude rather than balanced behavioral similarity across all features.

Within this representation:

- **Cluster 2 (red)** appears as a dispersed group of high-commission outliers, consistent with the behavioral profile of strategically valuable “top-tier” sellers.
- **Cluster 0 (blue)** and **Cluster 1 (green)** are compressed into a narrow region near the origin, making their separation visually ambiguous in the unscaled space.

This observation highlights one of the major limitations of directly applying distance-based clustering algorithms to highly skewed marketplace data without prior transformation.

In [5]:
display(HTML("<h3>Scaled K-Means Model Space</h3>"))
display(HTML(fig_scaled.to_html(
    full_html=False,
    include_plotlyjs=False
)))

### **Scaled Model Space — Effective Behavioral Separation After Transformation**

After applying the Yeo-Johnson transformation followed by standardization, the clustering structure becomes substantially clearer. In the transformed feature space, the three seller segments separate along a diagonal gradient extending from the lower-left region (low expected orders, low purchase probability, and low commission) toward the upper-right region (high values across all dimensions).

This pattern is behaviorally meaningful and indicates that the three predictive variables are positively correlated. More importantly, it demonstrates that the transformation process successfully reduced scale imbalance and enabled the K-Means algorithm to learn underlying seller behavior rather than merely detecting extreme numerical outliers.

Compared to the raw feature space, the scaled representation produces substantially more stable and interpretable clusters.

However, significant overlap is still observed between **Cluster 2 (red)** and **Cluster 1 (green)** in the middle region. The boundary is not sharp, meaning some sellers in the intermediate region may be assigned to the wrong cluster depending on the run.

## EXPORT


In [6]:
# ============================================================
# Export K-Means Seller Segmentation Results
# Uses existing notebook configs: NOTEBOOK_OUTPUT_DIR, HORIZON, K, FEATURES, FEATURE_GROUP_NAME
# ============================================================

from pathlib import Path
from datetime import datetime
import json
import pandas as pd
import joblib

# -------------------------
# Export config
# -------------------------
EXPORT_DIR = NOTEBOOK_OUTPUT_DIR / f"seller_segmentation_kmeans_k{K}_h{HORIZON}"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

CLUSTER_COL = "seller_cluster"

id_cols = [
    c for c in ["seller_id", "seller_unique_id", "shop_id", "user_id", "customer_id"]
    if c in df_kmeans.columns
]

metadata_cols = [
    c for c in ["kmeans_feature_group", "kmeans_horizon", "kmeans_k"]
    if c in df_kmeans.columns
]

print(f"Exporting to: {EXPORT_DIR.resolve()}")

# -------------------------
# 1. Full clustered dataset
# -------------------------
df_kmeans.to_csv(EXPORT_DIR / "seller_segmentation_kmeans_full.csv", index=False)

# -------------------------
# 2. Clean segmentation output
# -------------------------
segment_cols = id_cols + metadata_cols + FEATURES + [CLUSTER_COL]

if "seller_segment" in df_kmeans.columns:
    segment_cols.append("seller_segment")

segment_cols = [c for c in segment_cols if c in df_kmeans.columns]

seller_segments = df_kmeans[segment_cols].copy()
seller_segments.to_csv(EXPORT_DIR / "seller_segmentation_kmeans.csv", index=False)

# -------------------------
# 3. Cluster profile
# -------------------------
cluster_profile.to_csv(EXPORT_DIR / "cluster_profile.csv", index=False)

# -------------------------
# 4. Scaled model-space data
# -------------------------
scaled_cols = [f"scaled_{col}" for col in FEATURES]

model_space = pd.DataFrame(
    X_scaled,
    columns=scaled_cols,
    index=df_kmeans.index
)

model_space = pd.concat(
    [
        df_kmeans[id_cols] if id_cols else pd.DataFrame(index=df_kmeans.index),
        df_kmeans[[CLUSTER_COL]],
        model_space
    ],
    axis=1
)

model_space.to_csv(EXPORT_DIR / "kmeans_model_space_scaled.csv", index=False)

# -------------------------
# 5. Metrics
# -------------------------
metrics = {
    "exported_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "feature_group": FEATURE_GROUP_NAME,
    "horizon": HORIZON,
    "k": K,
    "features": FEATURES,
    "n_sellers": len(df_kmeans),
    "silhouette_score": float(silhouette),
    "davies_bouldin_score": float(davies_bouldin),
    "calinski_harabasz_score": float(calinski_harabasz),
}

with open(EXPORT_DIR / "kmeans_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=4, ensure_ascii=False)

# -------------------------
# 6. Excel report
# -------------------------
with pd.ExcelWriter(EXPORT_DIR / "seller_segmentation_kmeans_report.xlsx", engine="openpyxl") as writer:
    seller_segments.to_excel(writer, sheet_name="seller_segments", index=False)
    cluster_profile.to_excel(writer, sheet_name="cluster_profile", index=False)
    model_space.to_excel(writer, sheet_name="model_space_scaled", index=False)
    pd.DataFrame([metrics]).to_excel(writer, sheet_name="model_metrics", index=False)

# -------------------------
# 7. HTML plots
# -------------------------
if "fig_raw" in globals():
    fig_raw.write_html(EXPORT_DIR / "raw_business_space_3d.html")

if "fig_scaled" in globals():
    fig_scaled.write_html(EXPORT_DIR / "scaled_model_space_3d.html")

# -------------------------
# 8. Model artifacts
# -------------------------
ARTIFACT_DIR = EXPORT_DIR / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

joblib.dump(kmeans, ARTIFACT_DIR / "kmeans_model.joblib")
joblib.dump(transformer, ARTIFACT_DIR / "power_transformer.joblib")

with open(ARTIFACT_DIR / "features.json", "w", encoding="utf-8") as f:
    json.dump(FEATURES, f, indent=4, ensure_ascii=False)

# -------------------------
# Done
# -------------------------
print("Export completed.")
print(f"Main file: {EXPORT_DIR / 'seller_segmentation_kmeans.csv'}")
print(f"Report:    {EXPORT_DIR / 'seller_segmentation_kmeans_report.xlsx'}")
print(f"Folder:    {EXPORT_DIR.resolve()}")

display(cluster_profile)

Exporting to: E:\marketing for DS\marketing-driven-data-unlock\outputs\seller_segmentation_kmeans_k3_h60
Export completed.
Main file: e:\marketing for DS\marketing-driven-data-unlock\outputs\seller_segmentation_kmeans_k3_h60\seller_segmentation_kmeans.csv
Report:    e:\marketing for DS\marketing-driven-data-unlock\outputs\seller_segmentation_kmeans_k3_h60\seller_segmentation_kmeans_report.xlsx
Folder:    E:\marketing for DS\marketing-driven-data-unlock\outputs\seller_segmentation_kmeans_k3_h60


,seller_cluster,n_sellers,expected_orders_60d_mean,expected_orders_60d_median,p_purchase_60d_mean,p_purchase_60d_median,expected_commission_60d_mean,expected_commission_60d_median,expected_commission_60d_sum,seller_share,expected_commission_60d_share
2,2,857,13.877609,6.814062,0.985546,0.998902,283.500425,135.969629,242959.863867,0.381228,0.911484
1,1,552,1.414048,1.265127,0.700281,0.717793,37.214350,23.451205,20542.321191,0.245552,0.077066
0,0,839,0.158710,0.101584,0.136266,0.096595,3.637593,1.536463,3051.940941,0.373221,0.011450


## External Raw Profiling

In [10]:
# =========================
# PATH CONFIG
# =========================

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = ROOT / "data" / "raw"
OUT_DIR = ROOT / "notebooks" / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ITEMS_PATH = RAW_DIR / "olist_order_items_dataset.csv"
ORDERS_PATH = RAW_DIR / "olist_orders_dataset.csv"
SELLERS_PATH = RAW_DIR / "olist_sellers_dataset.csv"

for p in [ITEMS_PATH, ORDERS_PATH, SELLERS_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Missing file: {p}")
    
orders = pd.read_csv(ORDERS_PATH)
items = pd.read_csv(ITEMS_PATH)
sellers = pd.read_csv(SELLERS_PATH)

date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

items["shipping_limit_date"] = pd.to_datetime(
    items["shipping_limit_date"],
    errors="coerce"
)

print("orders:", orders.shape)
print("items:", items.shape)
print("sellers:", sellers.shape)

orders: (99441, 8)
items: (112650, 7)
sellers: (3095, 4)


In [19]:
# =========================
# TRUE EXTERNAL RAW FEATURES
# =========================

# Seller-order level để tránh duplicate order do 1 order có nhiều item
seller_order_items = (
    items
    .groupby(["seller_id", "order_id"])
    .agg(
        ext_items_in_order=("order_item_id", "count"),
        ext_products_in_order=("product_id", "nunique"),
        ext_freight_in_order=("freight_value", "sum"),
        ext_shipping_limit_date=("shipping_limit_date", "max"),
    )
    .reset_index()
)

raw = (
    seller_order_items
    .merge(orders, on="order_id", how="left")
    .merge(sellers, on="seller_id", how="left")
)

# Logistics
raw["ext_delivery_days"] = (
    raw["order_delivered_customer_date"] - raw["order_purchase_timestamp"]
).dt.total_seconds() / 86400

raw["ext_delay_vs_estimate_days"] = (
    raw["order_delivered_customer_date"] - raw["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

raw["ext_handoff_delay_days"] = (
    raw["order_delivered_carrier_date"] - raw["ext_shipping_limit_date"]
).dt.total_seconds() / 86400

raw["ext_approval_delay_hours"] = (
    raw["order_approved_at"] - raw["order_purchase_timestamp"]
).dt.total_seconds() / 3600

raw["ext_late_delivery_flag"] = (raw["ext_delay_vs_estimate_days"] > 0).astype(int)
raw["ext_late_handoff_flag"] = (raw["ext_handoff_delay_days"] > 0).astype(int)

# Order status
raw["ext_cancel_flag"] = raw["order_status"].eq("canceled").astype(int)
raw["ext_unavailable_flag"] = raw["order_status"].eq("unavailable").astype(int)
raw["ext_delivered_flag"] = raw["order_status"].eq("delivered").astype(int)

# Product diversity
seller_products = (
    items
    .groupby("seller_id")
    .agg(ext_n_unique_products=("product_id", "nunique"))
    .reset_index()
)

external_raw = (
    raw
    .groupby("seller_id")
    .agg(
        # Product assortment / basket structure
        ext_avg_items_per_order=("ext_items_in_order", "mean"),
        ext_multitem_order_rate=("ext_items_in_order", lambda x: (x > 1).mean()),
        ext_avg_products_per_order=("ext_products_in_order", "mean"),
        ext_multi_product_order_rate=("ext_products_in_order", lambda x: (x > 1).mean()),

        # Freight burden
        ext_avg_freight_per_order=("ext_freight_in_order", "mean"),
        ext_median_freight_per_order=("ext_freight_in_order", "median"),

        # Logistics performance
        ext_delivery_days_median=("ext_delivery_days", "median"),
        ext_delay_vs_estimate_days_median=("ext_delay_vs_estimate_days", "median"),
        ext_late_delivery_rate=("ext_late_delivery_flag", "mean"),
        ext_handoff_delay_days_median=("ext_handoff_delay_days", "median"),
        ext_late_handoff_rate=("ext_late_handoff_flag", "mean"),
        ext_approval_delay_hours_median=("ext_approval_delay_hours", "median"),

        # Order status quality
        ext_cancel_rate=("ext_cancel_flag", "mean"),
        ext_unavailable_rate=("ext_unavailable_flag", "mean"),
        ext_delivered_rate=("ext_delivered_flag", "mean"),

        # Geography
        seller_state=("seller_state", "first"),
        seller_city=("seller_city", "first"),
    )
    .reset_index()
    .merge(seller_products, on="seller_id", how="left")
    .replace([np.inf, -np.inf], np.nan)
)

CLUSTER_COL = "seller_cluster"

external_raw_df = (
    df_kmeans[["seller_id", CLUSTER_COL]]
    .drop_duplicates("seller_id")
    .merge(external_raw, on="seller_id", how="left")
)

display(external_raw_df.head())
print("external_raw_df:", external_raw_df.shape)

,seller_id,seller_cluster,ext_avg_items_per_order,ext_multitem_order_rate,ext_avg_products_per_order,ext_multi_product_order_rate,ext_avg_freight_per_order,ext_median_freight_per_order,ext_delivery_days_median,ext_delay_vs_estimate_days_median,ext_late_delivery_rate,ext_handoff_delay_days_median,ext_late_handoff_rate,ext_approval_delay_hours_median,ext_cancel_rate,ext_unavailable_rate,ext_delivered_rate,seller_state,seller_city,ext_n_unique_products
0,0015a82c2db000af6aaaf3ae2ecb0532,0,1.000000,0.000000,1.000000,0.000000,21.020000,21.02,10.747014,-12.301331,0.000000,-4.393970,0.000000,15.662778,0.0,0.0,1.000000,SP,santo andre,1
1,001cca7ae9ae17fb1caed9dfb1094831,2,1.195000,0.155000,1.005000,0.005000,44.270700,37.99,11.160208,-13.291690,0.065000,-3.662454,0.060000,0.297917,0.0,0.0,0.975000,ES,cariacica,11
2,002100f778ceb8431b7a1020ff7ab48f,2,1.078431,0.078431,1.078431,0.078431,15.561961,15.10,13.947975,-8.722789,0.176471,-2.978009,0.078431,0.290000,0.0,0.0,0.980392,SP,franca,24
3,003554e2dce176b5555353e4f3555ac8,0,1.000000,0.000000,1.000000,0.000000,19.380000,19.38,4.646806,-26.066794,0.000000,-5.476377,0.000000,0.310556,0.0,0.0,1.000000,GO,goiania,1
4,004c9cd9d87a3c30c522c48c4fc07416,2,1.075949,0.063291,1.018987,0.012658,22.476139,17.89,12.719201,-12.456383,0.082278,-5.591869,0.000000,0.276944,0.0,0.0,0.987342,SP,ibitinga,88


external_raw_df: (2248, 20)


## Numeric profiling + Kruskal-Wallis

In [ ]:
EXT_NUM_COLS = [
    "ext_n_unique_products",
    "ext_avg_items_per_order",
    "ext_multitem_order_rate",
    "ext_avg_products_per_order",
    "ext_multi_product_order_rate",
    "ext_avg_freight_per_order",
    "ext_median_freight_per_order",
    "ext_delivery_days_median",
    "ext_delay_vs_estimate_days_median",
    "ext_late_delivery_rate",
    "ext_handoff_delay_days_median",
    "ext_late_handoff_rate",
    "ext_approval_delay_hours_median",
    "ext_cancel_rate",
    "ext_unavailable_rate",
    "ext_delivered_rate",
]

EXT_NUM_COLS = [c for c in EXT_NUM_COLS if c in external_raw_df.columns]

external_profile = (
    external_raw_df
    .groupby(CLUSTER_COL)[EXT_NUM_COLS]
    .agg(["count", "mean", "median", "std"])
)

display(external_profile)

,seller_id,seller_cluster,ext_avg_items_per_order,ext_multitem_order_rate,ext_avg_products_per_order,ext_multi_product_order_rate,ext_avg_freight_per_order,ext_median_freight_per_order,ext_delivery_days_median,ext_delay_vs_estimate_days_median,ext_late_delivery_rate,ext_handoff_delay_days_median,ext_late_handoff_rate,ext_approval_delay_hours_median,ext_cancel_rate,ext_unavailable_rate,ext_delivered_rate,seller_state,seller_city,ext_n_unique_products
0,0015a82c2db000af6aaaf3ae2ecb0532,0,1.000000,0.000000,1.000000,0.000000,21.020000,21.02,10.747014,-12.301331,0.000000,-4.393970,0.000000,15.662778,0.0,0.0,1.000000,SP,santo andre,1
1,001cca7ae9ae17fb1caed9dfb1094831,2,1.195000,0.155000,1.005000,0.005000,44.270700,37.99,11.160208,-13.291690,0.065000,-3.662454,0.060000,0.297917,0.0,0.0,0.975000,ES,cariacica,11
2,002100f778ceb8431b7a1020ff7ab48f,2,1.078431,0.078431,1.078431,0.078431,15.561961,15.10,13.947975,-8.722789,0.176471,-2.978009,0.078431,0.290000,0.0,0.0,0.980392,SP,franca,24
3,003554e2dce176b5555353e4f3555ac8,0,1.000000,0.000000,1.000000,0.000000,19.380000,19.38,4.646806,-26.066794,0.000000,-5.476377,0.000000,0.310556,0.0,0.0,1.000000,GO,goiania,1
4,004c9cd9d87a3c30c522c48c4fc07416,2,1.075949,0.063291,1.018987,0.012658,22.476139,17.89,12.719201,-12.456383,0.082278,-5.591869,0.000000,0.276944,0.0,0.0,0.987342,SP,ibitinga,88


external_raw_df: (2248, 20)


ext_n_unique_products                               \
                               count       mean median        std   
seller_cluster                                                      
0                                839   4.835518    2.0  10.046984   
1                                552   6.297101    5.0   8.988324   
2                                857  28.217036   16.0  39.814488   

               ext_avg_items_per_order                                \
                                 count      mean    median       std   
seller_cluster                                                         
0                                  839  1.179451  1.000000  0.616022   
1                                  552  1.198907  1.000000  0.388455   
2                                  857  1.125228  1.069767  0.209311   

               ext_multitem_order_rate            ... ext_cancel_rate  \
                                 count      mean  ...          median   
seller_cluster                                    ...                   
0                                  839  0.102786  ...             0.0   
1                                  552  0.121055  ...             0.0   
2                                  857  0.086010  ...             0.0   

                         ext_unavailable_rate                             \
                     std                count      mean median       std   
seller_cluster                                                             
0               0.077944                  839  0.000238    0.0  0.006905   
1               0.031615                  552  0.000000    0.0  0.000000   
2               0.017632                  857  0.000003    0.0  0.000092   

               ext_delivered_rate                             
                            count      mean median       std  
seller_cluster                                                
0                             839  0.937220    1.0  0.146479  
1                             552  0.974688    1.0  0.068010  
2                             857  0.982457    1.0  0.033596  

[3 rows x 64 columns]

In [16]:
test_rows = []
clusters = sorted(external_raw_df[CLUSTER_COL].dropna().unique())

for col in EXT_NUM_COLS:
    groups = [
        external_raw_df.loc[external_raw_df[CLUSTER_COL] == c, col].dropna()
        for c in clusters
    ]
    groups = [g for g in groups if len(g) > 0]

    if len(groups) < 2:
        continue

    stat, p_value = kruskal(*groups)
    n = sum(len(g) for g in groups)
    k = len(groups)

    epsilon_squared = max(0, (stat - k + 1) / (n - k)) if n > k else np.nan

    test_rows.append({
        "variable": col,
        "statistic": stat,
        "p_value": p_value,
        "epsilon_squared": epsilon_squared,
    })

external_tests = (
    pd.DataFrame(test_rows)
    .sort_values(["epsilon_squared", "p_value"], ascending=[False, True])
)

display(external_tests)

,variable,statistic,p_value,epsilon_squared
0,ext_n_unique_products,904.108227,4.735796e-197,0.401830
9,ext_late_delivery_rate,219.469252,2.202203e-48,0.096868
3,ext_avg_products_per_order,193.501644,9.586319e-43,0.085301
4,ext_multi_product_order_rate,193.226021,1.100279e-42,0.085179
1,ext_avg_items_per_order,120.088363,8.378057e-27,0.052601
2,ext_multitem_order_rate,110.803171,8.697554e-25,0.048465
15,ext_delivered_rate,74.300619,7.342190e-17,0.032205
5,ext_avg_freight_per_order,62.555018,2.608258e-14,0.026973
7,ext_delivery_days_median,58.495083,1.985885e-13,0.025165
11,ext_late_handoff_rate,46.820930,6.807134e-11,0.019965


In [17]:
import plotly.express as px

median_profile = (
    external_raw_df
    .groupby(CLUSTER_COL)[EXT_NUM_COLS]
    .median()
)

profile_z = median_profile.T
profile_z = profile_z.sub(profile_z.mean(axis=1), axis=0)
profile_z = profile_z.div(profile_z.std(axis=1).replace(0, np.nan), axis=0)
profile_z = profile_z.dropna(how="all")

fig = px.imshow(
    profile_z,
    text_auto=".2f",
    aspect="auto",
    title="True External Raw Feature Profile by Seller Cluster - Median Z-Score",
    labels={
        "x": "Seller Cluster",
        "y": "True External Raw Variable",
        "color": "Relative Level",
    }
)

fig.show()

## Categorical profiling: state / city

In [20]:
from scipy.stats import chi2_contingency

cat_df = external_raw_df[[CLUSTER_COL, "seller_state", "seller_city"]].copy()

top_cities = cat_df["seller_city"].value_counts().head(30).index
cat_df["seller_city_grouped"] = np.where(
    cat_df["seller_city"].isin(top_cities),
    cat_df["seller_city"],
    "Other"
)

CAT_COLS = ["seller_state", "seller_city_grouped"]

cat_rows = []

for col in CAT_COLS:
    temp = cat_df[[CLUSTER_COL, col]].dropna()
    table = pd.crosstab(temp[CLUSTER_COL], temp[col])

    if table.shape[0] < 2 or table.shape[1] < 2:
        continue

    chi2, p_value, dof, expected = chi2_contingency(table)

    n = table.values.sum()
    r, c = table.shape
    cramers_v = np.sqrt(chi2 / (n * min(r - 1, c - 1)))

    cat_rows.append({
        "variable": col,
        "chi2": chi2,
        "p_value": p_value,
        "cramers_v": cramers_v,
        "n_categories": c,
    })

cat_tests = (
    pd.DataFrame(cat_rows)
    .sort_values(["cramers_v", "p_value"], ascending=[False, True])
)

display(cat_tests)

for col in CAT_COLS:
    print(f"\n=== {col} distribution by cluster ===")
    display(
        pd.crosstab(
            cat_df[CLUSTER_COL],
            cat_df[col],
            normalize="index"
        ).style.format("{:.1%}")
    )

,variable,chi2,p_value,cramers_v,n_categories
1,seller_city_grouped,79.090653,0.049932,0.132632,31
0,seller_state,41.393895,0.497424,0.095952,22



=== seller_state distribution by cluster ===


seller_state,AM,BA,CE,DF,ES,GO,MA,MG,MS,MT,PA,PB,PE,PI,PR,RJ,RN,RO,RS,SC,SE,SP
seller_cluster,,,,,,,,,,,,,,,,,,,,,,
0,0.1%,0.4%,0.5%,1.0%,0.5%,1.2%,0.0%,8.8%,0.2%,0.0%,0.0%,0.2%,0.1%,0.0%,11.7%,5.1%,0.1%,0.1%,4.4%,6.9%,0.1%,58.5%
1,0.0%,0.7%,0.4%,1.3%,0.2%,1.6%,0.0%,8.0%,0.4%,0.2%,0.2%,0.0%,0.4%,0.0%,13.0%,5.6%,0.4%,0.2%,5.3%,6.3%,0.2%,55.8%
2,0.0%,0.6%,0.4%,1.4%,0.7%,0.9%,0.1%,7.9%,0.0%,0.2%,0.0%,0.1%,0.5%,0.1%,11.0%,4.6%,0.0%,0.0%,2.7%,5.3%,0.0%,63.6%



=== seller_city_grouped distribution by cluster ===


seller_city_grouped,Other,barueri,bauru,belo horizonte,blumenau,brasilia,campinas,caxias do sul,curitiba,florianopolis,franca,goiania,guarulhos,ibitinga,joinville,limeira,londrina,marilia,maringa,mogi das cruzes,osasco,porto alegre,ribeirao preto,rio de janeiro,santo andre,sao bernardo do campo,sao caetano do sul,sao jose do rio preto,sao jose dos campos,sao paulo,sorocaba
seller_cluster,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,42.7%,0.7%,0.5%,2.4%,0.4%,0.8%,1.9%,0.6%,5.1%,0.7%,0.6%,0.7%,1.5%,1.0%,0.6%,0.4%,0.5%,0.8%,1.4%,0.6%,1.2%,1.1%,2.0%,3.3%,1.1%,0.2%,0.8%,0.8%,0.6%,23.8%,1.1%
1,45.1%,0.4%,0.4%,2.5%,0.9%,1.3%,1.8%,0.9%,4.7%,0.5%,0.7%,0.9%,0.9%,0.9%,1.1%,0.5%,1.1%,0.0%,1.6%,0.9%,1.3%,1.4%,2.5%,3.1%,1.6%,1.1%,0.4%,1.4%,0.2%,18.5%,1.3%
2,44.0%,0.2%,0.5%,2.0%,0.8%,1.3%,0.7%,0.4%,4.1%,0.4%,0.8%,0.6%,1.8%,3.4%,0.2%,0.8%,0.9%,0.5%,1.3%,0.5%,0.6%,0.6%,1.2%,2.5%,2.0%,1.3%,0.4%,1.2%,0.8%,23.9%,0.7%
